# Week 101, Day 3: Review a beginner's code

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w101d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('beginner.py', 'w', encoding='utf-8') as f:
    f.write('def total_km(walks):\n    """Add up the distance of every walk."""\n    total = 0\n    for i in range(len(walks)):\n        total = total + walks[i]["km"]\n    return total\n\n\ndef longest_walk(walks):\n    """Return the longest walk, or None."""\n    best = None\n    for walk in walks:\n        if best == None or walk["km"] > best["km"]:\n            best = walk\n    return best\n\n\ndef has_walks(walks):\n    if len(walks) > 0:\n        return True\n    else:\n        return False\n\n\ndef route_names(walks):\n    list = []\n    for walk in walks:\n        if walk["shared"] == True:\n            list.append(walk["route"])\n    return list\n\n\ndef load(path):\n    try:\n        with open(path) as f:\n            return f.read().splitlines()\n    except:\n        return []\n')

**Predict before you run:** what will this print?



In [ ]:
from beginner import has_walks, route_names, longest_walk

print(has_walks([]), route_names([{"route": "Ridge walk", "shared": True}]), longest_walk([]))

> **Key idea.** Review beginners to teach: name specific strengths first, choose two or three suggestions in order of importance, ask questions that let them find the improvement, explain why briefly, and end with an invitation. Watch for six habits: `range(len())`, `== None`, `== True`, returning `True`/`False` from an `if`, shadowed built-ins, and bare `except:`.

### Exercise: Finding the habits

Write `beginner_patterns(source)` with `ast`, returning sorted `(LINE, KIND)` pairs: `"range(len())"` for a `for` loop over `range(len(...))`; `"comparison with None"` and `"comparison with True or False"` for `==` or `!=` with `None` or a bool on either side; `"return the condition"` for an `if` whose only body statement and only `else` statement return `True` and `False` (either way round); `"shadowed built-in"` for assigning to a name that's a built-in; `"bare except"` for `except:` with no type.

In [ ]:
import ast
import builtins

def beginner_patterns(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_patterns():\n    \"six habits\"\n    assert beginner_patterns(open(\"beginner.py\").read()) == [\n        (4, \"range(len())\"), (13, \"comparison with None\"), (19, \"return the condition\"),\n        (26, \"shadowed built-in\"), (28, \"comparison with True or False\"), (37, \"bare except\")]\n    assert beginner_patterns(\"for walk in walks:\\n    print(walk)\\nif x is None:\\n    pass\\n\") == []\n    assert beginner_patterns(\"if None == x:\\n    pass\\n\") == [(1, \"comparison with None\")]")

### Exercise: What works

Write `strengths(source)`: a list of true, specific strengths, in this order: `"N of M functions have docstrings"` if at least one does; `"every function is short (at most 10 lines)"` if so; `"function names say what they do"` if every function name has at least 6 characters; `"files are closed with a with statement"` if there's a `with` statement.

In [ ]:
import ast

def strengths(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_strengths():\n    \"specific and true\"\n    assert strengths(open(\"beginner.py\").read()) == [\"2 of 5 functions have docstrings\", \"every function is short (at most 10 lines)\",\n                                                      \"files are closed with a with statement\"]\n    assert strengths(\"def go():\\n    pass\\n\") == [\"every function is short (at most 10 lines)\"]")

### Exercise: Two or three, most important first

Write `choose(findings, limit=3)`: from `(line, kind)` pairs, keep at most one finding of each kind, order them by importance (`bare except`, `shadowed built-in`, `comparison with None`, `comparison with True or False`, `range(len())`, `return the condition`), and return the first `limit` as `(line, kind)`.

In [ ]:
ORDER = ["bare except", "shadowed built-in", "comparison with None", "comparison with True or False", "range(len())", "return the condition"]

def choose(findings, limit=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_choose():\n    \"not twelve\"\n    findings = [(4, \"range(len())\"), (13, \"comparison with None\"), (19, \"return the condition\"), (26, \"shadowed built-in\"),\n                (28, \"comparison with True or False\"), (37, \"bare except\"), (40, \"comparison with None\")]\n    assert choose(findings) == [(37, \"bare except\"), (26, \"shadowed built-in\"), (13, \"comparison with None\")]\n    assert choose(findings, limit=1) == [(37, \"bare except\")] and choose([]) == []")

### Exercise: The message

Using the given `HINTS` (a question for each kind), write `feedback(name, praise, chosen)`: lines joined by newlines: `"Hi NAME, thanks for sharing this: it works, and it's clear what it's for."`, a blank line, `"What works well:"`, a `"- "` line for each of the first two strengths, a blank line, `"Things to try next:"`, a `"- line LINE: HINT"` line for each chosen finding, a blank line, and `"Which would you like to try first? I'm happy to pair on it."`. If there are no strengths, leave out that section (and its blank line).

In [ ]:
HINTS = {
    "bare except": "a bare except: also hides typos and Ctrl+C; which error do you expect here?",
    "shadowed built-in": "this name hides Python's own list(); what other name would describe it?",
    "comparison with None": "what does `is None` say that `== None` doesn't?",
    "comparison with True or False": "the value is already True or False; what happens if you use it directly?",
    "range(len())": "you only use i to get walks[i]; what would `for walk in walks` give you?",
    "return the condition": "the condition is already True or False; could you return it directly?",
}

def feedback(name, praise, chosen):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_message():\n    \"warm, short, specific\"\n    text = feedback(\"Sam\", [\"2 of 5 functions have docstrings\", \"every function is short (at most 10 lines)\", \"files are closed\"],\n                    [(37, \"bare except\"), (26, \"shadowed built-in\")])\n    assert text.splitlines() == [\n        \"Hi Sam, thanks for sharing this: it works, and it's clear what it's for.\", \"\",\n        \"What works well:\", \"- 2 of 5 functions have docstrings\", \"- every function is short (at most 10 lines)\", \"\",\n        \"Things to try next:\", f\"- line 37: {HINTS['bare except']}\", f\"- line 26: {HINTS['shadowed built-in']}\", \"\",\n        \"Which would you like to try first? I'm happy to pair on it.\"]\n    assert \"What works well\" not in feedback(\"Bo\", [], [(4, \"range(len())\")])")

### Exercise: The improved version, for comparison

For your own reference (not to send), rewrite `has_walks` and `route_names` without the habits: `has_walks(walks)` returns whether there are any walks in one line, and `route_names(walks)` returns the routes of shared walks with a comprehension.

In [ ]:
def has_walks(walks):
    if len(walks) > 0:
        return True
    else:
        return False

def route_names(walks):
    list = []
    for walk in walks:
        if walk["shared"] == True:
            list.append(walk["route"])
    return list

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\n\ndef test_rewrite():\n    \"same behaviour, clearer\"\n    walks = [{\"route\": \"Ridge walk\", \"shared\": True}, {\"route\": \"Lake loop\", \"shared\": False}]\n    assert has_walks(walks) is True and has_walks([]) is False and route_names(walks) == [\"Ridge walk\"]\n    tree = ast.parse(source)\n    assert not any(isinstance(n, ast.If) for n in ast.walk(tree)), \"no if needed\"\n    assert \"list =\" not in source and \"== True\" not in source")

### Bug hunt: None on the left

The check only looks at the right-hand side of the comparison, so `None == best` slips through and this prints `[]`. Check both sides so it prints `[1]`.

In [ ]:
import ast

source = "if None == best:\n    pass\n"
found = []
for node in ast.walk(ast.parse(source)):
    if isinstance(node, ast.Compare) and isinstance(node.ops[0], ast.Eq):
        right = node.comparators[0]
        if isinstance(right, ast.Constant) and right.value is None:
            found.append(node.lineno)
print(found)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"either side\"\n    expect_output(\"[1]\")")

### Bug hunt: Twelve suggestions

Sending every finding overwhelms a beginner; this sends all six and prints `6`. Send at most three so it prints `3`.

In [ ]:
findings = ["bare except", "shadowed built-in", "comparison with None", "comparison with True or False", "range(len())", "return the condition"]
suggestions = findings
print(len(suggestions))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"two or three\"\n    expect_output(\"3\")")

### Puzzle: Feedback in order

Arrange the parts of a message to a beginner. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the most suggestions to send a beginner at once (an int)
q = ___   # what to write instead of x == None
r = ___   # True or False: start a beginner review with what's wrong
s = ___   # hasattr(builtins, "list") after import builtins
t = ___   # the first habit to mention in beginner.py, by importance

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q.replace(\" \", \"\") == \"xisNone\" and r is False, \"p, q and r\"\n    assert s is True and t == \"bare except\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the whole review

Write `review_beginner(source, name, limit=2)` with today's helpers (given): find the patterns, choose up to `limit`, find the strengths, and return `(message, later)`: the `feedback` message, and the kinds found but not chosen, in order of importance (each once), to mention another time.

In [ ]:
import ast
import builtins

import ast, builtins

KINDS = ["bare except", "shadowed built-in", "comparison with None", "comparison with True or False", "range(len())", "return the condition"]

def beginner_patterns(source):
    found = []
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.For) and isinstance(node.iter, ast.Call) and getattr(node.iter.func, "id", None) == "range" \
                and node.iter.args and isinstance(node.iter.args[0], ast.Call) and getattr(node.iter.args[0].func, "id", None) == "len":
            found.append((node.lineno, "range(len())"))
        elif isinstance(node, ast.Compare):
            for op, right in zip(node.ops, node.comparators):
                if isinstance(op, (ast.Eq, ast.NotEq)):
                    sides = [node.left, right]
                    if any(isinstance(s, ast.Constant) and s.value is None for s in sides):
                        found.append((node.lineno, "comparison with None"))
                    elif any(isinstance(s, ast.Constant) and isinstance(s.value, bool) for s in sides):
                        found.append((node.lineno, "comparison with True or False"))
        elif isinstance(node, ast.If) and len(node.body) == 1 and len(node.orelse) == 1:
            a, b = node.body[0], node.orelse[0]
            if isinstance(a, ast.Return) and isinstance(b, ast.Return) and isinstance(a.value, ast.Constant) \
                    and isinstance(b.value, ast.Constant) and {a.value.value, b.value.value} == {True, False}:
                found.append((node.lineno, "return the condition"))
        elif isinstance(node, ast.Assign):
            for t in node.targets:
                if isinstance(t, ast.Name) and hasattr(builtins, t.id):
                    found.append((node.lineno, "shadowed built-in"))
        elif isinstance(node, ast.ExceptHandler) and node.type is None:
            found.append((node.lineno, "bare except"))
    return sorted(found)

ORDER = ["bare except", "shadowed built-in", "comparison with None", "comparison with True or False", "range(len())", "return the condition"]

def choose(findings, limit=3):
    first = {}
    for line, kind in sorted(findings):
        first.setdefault(kind, line)
    ranked = sorted(first.items(), key=lambda kv: ORDER.index(kv[0]))
    return [(line, kind) for kind, line in ranked[:limit]]

def strengths(source):
    tree = ast.parse(source)
    funcs = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
    found = []
    documented = sum(1 for f in funcs if ast.get_docstring(f))
    if documented:
        found.append(f"{documented} of {len(funcs)} functions have docstrings")
    if funcs and all(f.end_lineno - f.lineno + 1 <= 10 for f in funcs):
        found.append("every function is short (at most 10 lines)")
    if funcs and all(len(f.name) >= 6 for f in funcs):
        found.append("function names say what they do")
    if any(isinstance(n, ast.With) for n in ast.walk(tree)):
        found.append("files are closed with a with statement")
    return found

HINTS = {
    "bare except": "a bare except: also hides typos and Ctrl+C; which error do you expect here?",
    "shadowed built-in": "this name hides Python's own list(); what other name would describe it?",
    "comparison with None": "what does `is None` say that `== None` doesn't?",
    "comparison with True or False": "the value is already True or False; what happens if you use it directly?",
    "range(len())": "you only use i to get walks[i]; what would `for walk in walks` give you?",
    "return the condition": "the condition is already True or False; could you return it directly?",
}

def feedback(name, praise, chosen):
    lines = [f"Hi {name}, thanks for sharing this: it works, and it's clear what it's for.", ""]
    if praise:
        lines += ["What works well:"] + [f"- {p}" for p in praise[:2]] + [""]
    lines += ["Things to try next:"] + [f"- line {line}: {HINTS[kind]}" for line, kind in chosen]
    lines += ["", "Which would you like to try first? I'm happy to pair on it."]
    return "\n".join(lines)

def review_beginner(source, name, limit=2):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_review():\n    \"beginner.py\"\n    message, later = review_beginner(open(\"beginner.py\").read(), \"Sam\")\n    lines = message.splitlines()\n    assert lines[3] == \"- 2 of 5 functions have docstrings\" and lines[7] == f\"- line 37: {HINTS['bare except']}\"\n    assert lines[8] == f\"- line 26: {HINTS['shadowed built-in']}\" and len(lines) == 11\n    assert later == [\"comparison with None\", \"comparison with True or False\", \"range(len())\", \"return the condition\"]\n\ndef test_clean():\n    \"nothing to change\"\n    message, later = review_beginner('def greet_walker(name):\\n    \"\"\"Say hello.\"\"\"\\n    return \"Hello \" + name\\n', \"Bo\")\n    assert \"Things to try next:\" in message and later == []")

## Warm-up

Everything in `beginner.py` works. That matters: a review that opens with "this is wrong" would be untrue, and would discourage someone who has just made something work for the first time. The habits in it are worth changing, but as the next step of their learning, not as errors.

## Reviewing to teach

- **Start with what works**, specifically and truthfully: "the docstrings make each function's job clear", not "great job!".
- **Choose two or three** suggestions, most important first (Week 97's severity order still applies: a bare `except:` that hides mistakes beats a style point). Note the rest for next time.
- **Ask rather than tell** where you can: "you only use `i` to get `walks[i]`; what would `for walk in walks` give you?" lets them discover the improvement, which they'll remember.
- **Explain why**, briefly: a habit with a reason sticks; a rule without one doesn't.
- **End with a question** that invites a reply.

## Six habits

- `for i in range(len(walks))` when only `walks[i]` is used: loop over the items.
- `x == None`: use `x is None`.
- `x == True` or `x == False`: use `x` or `not x`.
- `if condition: return True else: return False`: return the condition.
- Naming a variable `list`, `dict`, `sum` or another built-in: it hides the built-in.
- A bare `except:`: catch the error you expect.

The syntax tree finds all six; Week 88's review tools did the same for Waymark.

## The message

A short, warm structure works: thanks, two specific strengths, two or three suggestions as questions with the line they refer to, and an invitation. Keep it short enough to read in a minute. And offer to pair on the first change (tomorrow's lesson).

## Practice

## Quiz

## Daily challenge

## Your review

1. **Find a beginner's code** to review: a friend learning Python, a question on a beginners' forum, or your own Week 2 exercises.
2. **Run your tools** on it, then read it yourself: the tools miss things, and only you can judge what this learner is ready for.
3. **Write the message**: two strengths, two or three questions, an invitation. Read it aloud before sending: would it encourage you?
4. **Follow up** on whatever they reply.

## Before you go

Tomorrow is a mentor session: working alongside a learner, asking good questions, and agreeing goals they can reach.